# Exploring Corpus

This notebook explores and describes the corpus.

- What is the variation in header names?
- What is the frequency of words in sentences (group Parkinson vs non-Parkinson)?
- How many words are there in a sentence?
- How many sentences are there in a paper?

## Initialisation

In [ ]:
# meta
__author__ ="Jennefer Beenen"
__version__ = "1.0"
__email__ = "j.beenen@pl.hanze.nl"
__status__ = "Development"
# __date__ = "2025-05-06"

In [ ]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# https://amueller.github.io/word_cloud/
from wordcloud import WordCloud
from wordcloud import STOPWORDS

from sentence_transformers import SentenceTransformer

### Settings

In [ ]:
# load data
filename = 'corpus_free_3600_250606'
df = pd.read_csv(f'../data/corpus/{filename}.csv')
df.info()

### Functions

In [ ]:
def unique_values_per_column(df: pd.DataFrame):
    """Prints the unique values found in each column of a data frame."""
    for column in df.columns:
        if df[column].nunique() < 25:
            print(f"'{column}' ({df[column].nunique()}):\n{df[column].unique()}\n")
        else:
            print(f"'{column}' ({df[column].nunique()}): (only first 25 values are shown)\n{df[column].unique()[:25]}\n")

def get_sentences(serie, keywords: list, inverse: bool = False) -> list:
    """Get sentences from serie that contains the keyword, or sentences without the keyword by setting `inverse` to `True`."""
    # Make all sentence_text lower case & remove puntuations
    # https://www.geeksforgeeks.org/python/generating-word-cloud-python/
    serie = serie.str.lower().str.strip('.?!')

    if inverse:
        # Search for sentences that exclude `keywords`
        # Note: Copilot brought to my attention that `serie.str.contains` exists
        # and that it can handle a list by creating a regex expression with "|".join()
        sentences = serie.loc[~serie.str.contains("|".join(keywords))]
        # Report number of sentences found
        print(f"{len(sentences)} sentences were found without '{keywords}'.")   
    else:
        # Search for sentences that include `keywords`
        sentences = serie.loc[serie.str.contains("|".join(keywords))]
        # Report number of sentences found
        print(f"{len(sentences)} sentences were found with '{keywords}'.")

    # (Ready to create wordcloud)
    return sentences

def create_wordcloud(sentences: list):
    """Create WordCloud object, 16:9 ratio, top 100 words."""
    # Create WordCloud object, 16:9 ratio, top 100 words
    wordcloud = WordCloud(
        width=1600,
        height=900,
        max_words= 100,
        stopwords= STOPWORDS.update(['one', 'two', 'three', 'first', 'study']),
        colormap= 'PiYG'
    ).generate(" ".join(sentences)) # join words to form one long 'sentence'

    # Plot
    plt.imshow(wordcloud)
    plt.axis('off')
    plt.show()

### Replace 'PD' with "parkinson's disease"

I'm expecting Parkinson's Disease to be often abbrevated to PD.

In [ ]:
# Current state:
df.head()

In [ ]:
# Check if "PD" can mean Parkinson.
PD_indexes = df['sentence_text'].str.findall("PD").str.len() > 0
# Show
df.loc[PD_indexes, 'sentence_text'].values

As shown above. PD often used as an abbreviation for parkinson's disease.
(also see https://www.ncbi.nlm.nih.gov/books/NBK536722/)

In [ ]:
# Replace " PD " with " parkinson disease ".
df['sentence_text'] = df['sentence_text'].str.replace("PD", "parkinson's disease", case= True)
# Show if correction was successful 
df.loc[PD_indexes, 'sentence_text'].values

Note that the code above is also used in `corpus2vectors.py` before embedding sentences.

In [ ]:
# Make all sentence_text lower case & remove puntuations
# https://www.geeksforgeeks.org/python/generating-word-cloud-python/
df['sentence_text'] = df['sentence_text'].str.lower().str.strip('.?!')

In [ ]:
# Search for sentences that include 'Parkinson'
sentences = df.loc[df['sentence_text'].str.findall("parkinson's disease").str.len() > 0, 'sentence_text'].values
sentences

Note that total number of sentences has now increased: 

3843: when searching only on "PD".           

4154: now together with "parkinson's disease".

## Frequency of header names

In [ ]:
# empty sentence_text
df[df['sentence_text'].isna()]

In [ ]:
# explore unique values
unique_values_per_column(df)

`head_name` also appears to contain sentences. Example: "unlike emem/f12 media which contained 1.5 mm pyruvate, the rpmi medium did not contain pyruvate"

In [ ]:
# turn all head_name s to lower case
df['head_name'] = df['head_name'].str.lower()

In [ ]:
# plot the distribution of the 'head_name' column
df.groupby(['head_name'])['paper_id'].nunique().sort_values(ascending=False).head(20).plot(kind='bar', figsize=(10, 6))

# annotate value above each bar
for i, v in enumerate(df.groupby(['head_name'])['paper_id'].nunique().sort_values(ascending=False).head(20)):
    plt.text(i, v + 0.5, str(v), ha='center', va='bottom')

plt.title('Distribution of head_name')
plt.ylabel('Number of unique paper_id')
# plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Frequency words (wordcloud)

Wordcloud python library works the best with English words (for auto removing stopwords) and removes "'s" from text. See [documentation](https://amueller.github.io/word_cloud/_modules/wordcloud/wordcloud.html#WordCloud).

**NOTE:** Keep in mind that the processed papers are already biased to (pesticides AND parkinson's disease).

### Example

In [ ]:
# Example from Michiel Noback

# Choose text myself:
# I Want To Live - Borislav Slavov (Baldur's Gate 3)
words = """
I feel your breath upon my neck
A soft caress as cold as death (Cold as death)
I didn't know you well back then
I blame it all on luck and vain (Luck and vain)
Your blood like wine, I wanted in
Oh darling, get me drunk and make me feel

It's not my fault
I'm not to blame
These ain't my sins
I broke my chains
There's more to do
And I still want to live

I feel your breath, upon my neck
A soft caress, as cold as death (Cold as death)
I feel your heart-beat in my soul
Our futures bound, our bodies know (Bodies know)
Your blood like wine, I wanted in
Oh darling get me drunk, invite me in

It's not my fault
I'm not to blame
These ain't my sins
I broke my chains
There's more to do
If I can only live

I can't go yet
Don't let me die
I'll never stop
Until I'm done
But just tonight
Maybe I'll rest in peace

I feel your breath upon my neck
A soft caress as cold as death (Cold as death)
I hear your heart-beat in my soul
Our endings bound, our bodies know

I can't go yet
Don't let me die
I want to live
My only one
There's more to do, if we can only live
The clock won't stop and this is what we get
"""

wordcloud = WordCloud(width=600, height=400).generate(words)
plt.imshow(wordcloud)
plt.axis('off')
plt.show()

### Combined Parkinson's disease & Pesticides

In [ ]:
# Sentences with "pesticides", "parkinson's disease"
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        ["pesticides", "parkinson's disease"],
    )
)

In [ ]:
# Sentences without "pesticides", "parkinson's disease"
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        ["pesticides", "parkinson's disease"],
        inverse= True
    )
)

### Parkinson's disease

In [ ]:
# with
create_wordcloud(
    get_sentences(
        df["sentence_text"], 
        ["parkinson's disease"]
    )
)

In [ ]:
# without
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        ["parkinson's disease"],
        inverse= True
    )
)

### Pesticides

In [ ]:
# with
create_wordcloud(
    get_sentences(
        df["sentence_text"], 
        ["pesticides"]
    )
)

In [ ]:
# without
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        ["pesticides"],
        inverse= True
    )
)

### Various other keywords

In [ ]:
# with Mitochondria
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        "mitochondria"
    )
)

In [ ]:
# with Microglia
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        "microglia"
    )
)

In [ ]:
# with Paraquat
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        "paraquat"
    )
)

In [ ]:
# with Chlorpyrifos
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        "chlorpyrifos"
    )
)

In [ ]:
# with Acetamiprid
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        "acetamiprid"
    )
)

In [ ]:
# with Pendimethalin
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        "pendimethalin"
    )
)

## Number of words per sentence

In order to choose the right model for determining sentence similarity, it is helpfull to know general properties (e.g. number of words per sentence). Some transformers can only handle a limited amount of words per sentence.

"A common value for BERT-based models are 512 tokens, which corresponds to about 300-400 words (for English)." https://www.sbert.net/examples/sentence_transformer/applications/computing-embeddings/README.html#input-sequence-length 

In [ ]:
# Load model of choice
model_st = SentenceTransformer('NeuML/pubmedbert-base-embeddings')

# Max number of tokens per sentence.
# Longer texts will be truncated to the first model.max_seq_length tokens
# https://www.sbert.net/examples/sentence_transformer/applications/computing-embeddings/README.html#input-sequence-length 
model_st.max_seq_length

#### Check word count distribution of sentences in our corpus:

The following results have the description for experiment `free_3600_250606`.

In [ ]:
# Create wordcount serie
s_word_count = df['sentence_text'].str.split(' ').map(len)

In [ ]:
# Plot histogram
s_word_count.plot.hist(bins= 25)

plt.title("Words per sentence distribution")
plt.xlabel("Number of words")
plt.show()

In [ ]:
s_word_count.describe()

Of the 18305 sentences, an average sentence contain 24 words, where the longest sentence is 174 words long.

To use 'Sentence-BERT' for spaCy the sentences need to be shorter than 128.

In [ ]:
# Sentences that are longer than 128 words.
s_word_count[s_word_count >128]

There are 5 sentences that are longer.

In [ ]:
i_long_sentences = s_word_count[s_word_count >128].index

In [ ]:
df.loc[i_long_sentences]

The long sentences often appear in Material and Methods (3/5), or in the Discussion (2/5).

## Number of sentences per paper

The absolute number of sentences per paper

In [ ]:
# Number of headers, paragraphs, and sentences per paper.
df.groupby('paper_id')[['head_id', 'paragraph_id', 'sentence_text']].nunique()

#### Number of sentences per paper globally described

In [ ]:
df.groupby('paper_id')[['sentence_text']].nunique().plot.hist(bins= 25)
plt.title("Distribution of total number of sentences per paper")
plt.xlabel('Number of sentences')
plt.show()

In [ ]:
df.groupby('paper_id')[['sentence_text']].nunique().describe()

Of the 81 papers, an average sentence contain 225 sentences, where the longest paper is 1127 sentences long.